In [3]:
# ============================================================
# PB-06 | PROTOTIPO INTERACTIVO EN UNA SOLA CELDA
# Incluye instalación de ipywidgets, carga de datos e interfaz
# ============================================================

import sys
import subprocess
import importlib

# ------------------------------------------------------------
# 1. INSTALACIÓN AUTOMÁTICA DE LIBRERÍAS FALTANTES
# ------------------------------------------------------------
def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    try:
        importlib.import_module(import_name)
        print(f"✅ {package_name} ya está instalado")
    except ImportError:
        print(f"📦 Instalando {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"✅ {package_name} instalado correctamente")

ensure_package("ipywidgets")
ensure_package("seaborn")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("numpy")

# ------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 5)

print("✅ Librerías cargadas correctamente")

# ------------------------------------------------------------
# 3. CONFIGURACIÓN DEL DATASET
# ------------------------------------------------------------
# Cambia esta ruta si tu archivo está en otro lugar
DATA_PATH = Path("../data/raw/dataset_pucp.csv")

# Cambia este nombre si tu target no se llama Revenue
target_col = "Revenue"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset en: {DATA_PATH.resolve()}\n"
        f"Corrige DATA_PATH o ejecuta dvc pull."
    )

df = pd.read_csv(DATA_PATH)
print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"📍 Ruta usada: {DATA_PATH.resolve()}")

# ------------------------------------------------------------
# 4. PREPARACIÓN DE VARIABLES
# ------------------------------------------------------------
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

if target_col in numeric_cols:
    numeric_cols.remove(target_col)

target_values = ["Todos"]
if target_col in df.columns:
    target_values += sorted(df[target_col].dropna().astype(str).unique().tolist())

print(f"✅ Variables numéricas detectadas: {len(numeric_cols)}")
print(f"✅ Variables categóricas detectadas: {len(categorical_cols)}")
print(f"✅ Valores del target detectados: {target_values if target_col in df.columns else 'Target no encontrado'}")

# ------------------------------------------------------------
# 5. FUNCIONES AUXILIARES
# ------------------------------------------------------------
def filtrar_outliers_iqr(dataframe, column):
    serie = dataframe[column].dropna()
    if serie.empty:
        return dataframe

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    if iqr == 0:
        return dataframe

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return dataframe[(dataframe[column] >= lower) & (dataframe[column] <= upper)]


def plot_interactivo(variable, tipo_grafico, bins, filtrar_target, quitar_outliers, mostrar_resumen):
    data = df.copy()

    if variable is None or variable not in data.columns:
        print("❌ No hay una variable válida para visualizar.")
        return

    if target_col in data.columns and filtrar_target != "Todos":
        data = data[data[target_col].astype(str) == str(filtrar_target)]

    if quitar_outliers:
        data = filtrar_outliers_iqr(data, variable)

    if data.empty:
        print("⚠️ No hay datos luego de aplicar los filtros.")
        return

    if mostrar_resumen:
        display(Markdown(
            f"""
### Resumen actual
- **Variable:** `{variable}`
- **Tipo de gráfico:** `{tipo_grafico}`
- **Bins:** `{bins}`
- **Filtro target:** `{filtrar_target}`
- **Quitar outliers:** `{quitar_outliers}`
- **Registros mostrados:** `{len(data)}`
"""
        ))

    fig, ax = plt.subplots(figsize=(11, 5))

    if tipo_grafico == "Histograma":
        sns.histplot(data=data, x=variable, bins=bins, kde=True, ax=ax)
        ax.set_title(f"Histograma de {variable}")
        ax.set_xlabel(variable)
        ax.set_ylabel("Frecuencia")

    elif tipo_grafico == "Boxplot":
        sns.boxplot(data=data, y=variable, ax=ax)
        ax.set_title(f"Boxplot de {variable}")
        ax.set_ylabel(variable)

    elif tipo_grafico == "Boxplot por target":
        if target_col not in data.columns:
            print(f"⚠️ No existe la columna target '{target_col}'")
            return
        sns.boxplot(data=data, x=target_col, y=variable, ax=ax)
        ax.set_title(f"{variable} por {target_col}")
        ax.set_xlabel(target_col)
        ax.set_ylabel(variable)

    elif tipo_grafico == "Violin por target":
        if target_col not in data.columns:
            print(f"⚠️ No existe la columna target '{target_col}'")
            return
        sns.violinplot(data=data, x=target_col, y=variable, ax=ax)
        ax.set_title(f"Distribución de {variable} por {target_col}")
        ax.set_xlabel(target_col)
        ax.set_ylabel(variable)

    plt.tight_layout()
    plt.show()


def plot_target(tipo_target_plot):
    if target_col not in df.columns:
        print(f"❌ No existe la columna target '{target_col}'")
        return

    conteos = df[target_col].value_counts(dropna=False)
    porcentajes = df[target_col].value_counts(normalize=True, dropna=False) * 100

    fig, ax = plt.subplots(figsize=(8, 5))

    if tipo_target_plot == "Barras":
        sns.barplot(x=conteos.index.astype(str), y=conteos.values, ax=ax)
        ax.set_title(f"Distribución del target: {target_col}")
        ax.set_xlabel(target_col)
        ax.set_ylabel("Frecuencia")

        for i, v in enumerate(conteos.values):
            ax.text(i, v, str(v), ha="center", va="bottom")

    elif tipo_target_plot == "Pie":
        ax.pie(
            conteos.values,
            labels=conteos.index.astype(str),
            autopct="%1.1f%%",
            startangle=90
        )
        ax.set_title(f"Porcentaje del target: {target_col}")

    display(pd.DataFrame({
        "Frecuencia": conteos,
        "Porcentaje": porcentajes.round(2)
    }))

    plt.tight_layout()
    plt.show()


def plot_relacion(x_var, y_var, color_by_target, alpha):
    if x_var is None or y_var is None:
        print("❌ Selecciona variables válidas.")
        return

    if x_var not in df.columns or y_var not in df.columns:
        print("❌ Una o ambas variables no existen.")
        return

    if x_var == y_var:
        print("⚠️ Selecciona dos variables distintas.")
        return

    fig, ax = plt.subplots(figsize=(10, 6))

    if color_by_target and target_col in df.columns:
        sns.scatterplot(data=df, x=x_var, y=y_var, hue=target_col, alpha=alpha, ax=ax)
        ax.legend(title=target_col, bbox_to_anchor=(1.05, 1), loc="upper left")
    else:
        sns.scatterplot(data=df, x=x_var, y=y_var, alpha=alpha, ax=ax)

    ax.set_title(f"Relación entre {x_var} y {y_var}")
    ax.set_xlabel(x_var)
    ax.set_ylabel(y_var)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 6. WIDGETS | EXPLORADOR PRINCIPAL
# ------------------------------------------------------------
variable_dropdown = widgets.Dropdown(
    options=numeric_cols,
    value=numeric_cols[0] if numeric_cols else None,
    description="Variable:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

tipo_radio = widgets.RadioButtons(
    options=["Histograma", "Boxplot", "Boxplot por target", "Violin por target"],
    value="Histograma",
    description="Gráfico:",
    style={"description_width": "initial"}
)

bins_slider = widgets.IntSlider(
    value=30,
    min=5,
    max=60,
    step=5,
    description="Bins:",
    style={"description_width": "initial"},
    continuous_update=False
)

target_dropdown = widgets.Dropdown(
    options=target_values,
    value="Todos",
    description="Filtro target:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

outliers_checkbox = widgets.Checkbox(
    value=False,
    description="Quitar outliers"
)

resumen_checkbox = widgets.Checkbox(
    value=True,
    description="Mostrar resumen"
)

out_principal = widgets.interactive_output(
    plot_interactivo,
    {
        "variable": variable_dropdown,
        "tipo_grafico": tipo_radio,
        "bins": bins_slider,
        "filtrar_target": target_dropdown,
        "quitar_outliers": outliers_checkbox,
        "mostrar_resumen": resumen_checkbox
    }
)

ui_principal = widgets.VBox([
    widgets.HTML("<h2>Explorador interactivo de variables numéricas</h2>"),
    widgets.HBox([variable_dropdown, target_dropdown]),
    widgets.HBox([bins_slider]),
    widgets.HBox([outliers_checkbox, resumen_checkbox]),
    tipo_radio
])

# ------------------------------------------------------------
# 7. WIDGETS | EXPLORADOR DEL TARGET
# ------------------------------------------------------------
target_radio = widgets.RadioButtons(
    options=["Barras", "Pie"],
    value="Barras",
    description="Vista target:",
    style={"description_width": "initial"}
)

out_target = widgets.interactive_output(
    plot_target,
    {"tipo_target_plot": target_radio}
)

ui_target = widgets.VBox([
    widgets.HTML("<h2>Explorador del balance de clases</h2>"),
    target_radio
])

# ------------------------------------------------------------
# 8. WIDGETS | RELACIÓN ENTRE VARIABLES
# ------------------------------------------------------------
x_dropdown = widgets.Dropdown(
    options=numeric_cols,
    value=numeric_cols[0] if len(numeric_cols) > 0 else None,
    description="Eje X:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

y_dropdown = widgets.Dropdown(
    options=numeric_cols,
    value=numeric_cols[1] if len(numeric_cols) > 1 else (numeric_cols[0] if numeric_cols else None),
    description="Eje Y:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

color_checkbox = widgets.Checkbox(
    value=True,
    description=f"Colorear por {target_col}"
)

alpha_slider = widgets.FloatSlider(
    value=0.6,
    min=0.2,
    max=1.0,
    step=0.1,
    description="Transparencia:",
    style={"description_width": "initial"},
    continuous_update=False
)

out_relacion = widgets.interactive_output(
    plot_relacion,
    {
        "x_var": x_dropdown,
        "y_var": y_dropdown,
        "color_by_target": color_checkbox,
        "alpha": alpha_slider
    }
)

ui_relacion = widgets.VBox([
    widgets.HTML("<h2>Explorador de relación entre variables</h2>"),
    widgets.HBox([x_dropdown, y_dropdown]),
    widgets.HBox([color_checkbox, alpha_slider])
])

# ------------------------------------------------------------
# 9. MOSTRAR TODO EN UNA SOLA “PÁGINA”
# ------------------------------------------------------------
pagina = widgets.VBox([
    widgets.HTML("<h1>PB-06 | Prototipo interactivo de exploración</h1>"),
    widgets.HTML("<p>Interfaz en una sola celda con ipywidgets.</p>"),
    ui_principal,
    out_principal,
    widgets.HTML("<hr>"),
    ui_target,
    out_target,
    widgets.HTML("<hr>"),
    ui_relacion,
    out_relacion
])

display(pagina)

✅ ipywidgets ya está instalado
✅ seaborn ya está instalado
✅ pandas ya está instalado
✅ matplotlib ya está instalado
✅ numpy ya está instalado
✅ Librerías cargadas correctamente
✅ Dataset cargado: 12330 filas x 18 columnas
📍 Ruta usada: C:\Users\Admin\dp261-g4\data\raw\dataset_pucp.csv
✅ Variables numéricas detectadas: 14
✅ Variables categóricas detectadas: 4
✅ Valores del target detectados: ['Todos', 'False', 'True']
